                                                             PROMPT ENGINEERING

In [ ]:
# Loading a Text Generation Model

In [ ]:
# Install or upgrade the required libraries to specific versions
!pip install -U \
transformers==4.46.3 \       # Hugging Face Transformers library
huggingface_hub==0.26.2 \    # Library for accessing Hugging Face Hub
accelerate==1.1.1 \           # Helps efficiently run models on CPU/GPU
tokenizers==0.20.3 \          # Provides fast tokenization for Transformer models
safetensors                    # Secure and efficient format for storing model weights

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
"microsoft/Phi-3-mini-4k-instruct",
device_map="cuda",
torch_dtype="auto",
trust_remote_code=True)

tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

# Create a pipeline
pipe = pipeline(
"text-generation",
model=model,
tokenizer=tokenizer,
return_full_text=False,
max_new_tokens=500,
do_sample=False)

# Prompt
messages = [
{"role": "user", "content": "Create a funny joke about chickens."}
]

# Pipeline automatically converts the prompt into prompt template
# So dont need to apply the prompt template
# Generate the output
output = pipe(messages)
print(output[0]["generated_text"])

In [ ]:
# Applying chat template to see how the model recieves the prompt that converted by pipeline
# This is useful when you are not using pipeline
# pipe.tokenizer = above we created pipeline so take tokenizer which is used by the pipeline
# In absence of pipeline you can directly use the tokenizer
# tokenizer.apply_chat_template = Apply the chat template which has in tokenizer.
# This converts the conversation into models expected format
prompt = pipe.tokenizer.apply_chat_template(messages,         # apply template to messages

                                           # True = conversation is converted into token ids
                                           tokenize=False)    # Just Format the conversation dont change into token ids
print(prompt)

In [ ]:
#   Controlling Model Output

In [ ]:
 """ We can control the output of the model by changing the paramters
 do_sample=False  :
                  the model doesnt uses sampling .The next token is selected based on the probability . Same output every time
 do_sample=true   :
                  If sampling is enabled two paramteres effect the output 1.Temperature  2.top_p

 TEMPERATURE :  It controls the randomness
    Low = Model selects the high probability token .Output will be more accurate
    High = Selects the low probability token .Output will be creative

TOP_P : It controls how many tokens to select
   Low = Only small set of tokens are selected . Output is less creative
   High = Large set of tokens .Output is more creative



In [ ]:
# Using a high temperature
output = pipe(messages,
              do_sample=True,
              temperature=1)
print(output[0]["generated_text"])

In [ ]:
# Using a high top_p
output = pipe(messages, do_sample=
True
, top_p=1)
print(output[0]["generated_text"])

In [ ]:
""" ADAVANCE PROMPT ENGINEERING

The prompt need not to contain only one instruction
We can you many different components to get the exact output what we want
Each component gives the llm more guidance
1. Persona = tells the model to act as which person based on who should access the output
2. Instruction = what the model should do
3. Context = It gives the additional informartion about your task
4. data_format = How the answer should look
5. audience = Who want this output (which category)
6. tone = How the output sounds (professional , comedy)
7. data = Model performs the task on that given data
"""

In [ ]:
# Prompt components
persona = "You are an expert in Large Language models. You excel at breaking down complex papers into digestible summaries.\n"

instruction = "Summarize the key findings of the paper provided.\n"

context = "Your summary should extract the most crucial points that can help researchers quickly understand the most vital information of the paper.\n"

data_format = "Create a bullet-point summary that outlines the method. Follow this up with a concise paragraph that encapsulates the main results.\n"

audience = "The summary is designed for busy researchers that quickly need to grasp the newest trends in Large Language Models.\n"

tone = "The tone should be professional and clear.\n"

text = "Large Language Models are becoming increasingly popular fortext generation, summarization, translation, and question answering."
data = f"Text to summarize: {text}"

# The full prompt - remove and add pieces to view its impact on the generated output
query = persona + instruction + context + data_format + audience + tone + data

messages = [
{"role": "user", "content": query}   # role = who is asking , content = What they asking
]

# Pipeline generates the answer for the query and stored in output
output = pipe(query)

# Output contains a list , select the first generated response
print(output[0]["generated_text"])


In [ ]:
""" In context learning = providing the examples
        === TYPES===
Zero shot = prompt with no examples
one shot = prompt with one example
few shot = prompt with few examples
"""

In [ ]:
#   ONE SHOT
# Here prompt is written like a real conversation

In [ ]:
one_shot_prompt = [
{
    # User asks the question
"role": "user",
"content": "A 'Gigamuru' is a type of Japanese musical instrument. An example of a sentence that uses the word Gigamuru is:"
},
{
    # Model will reply like this
"role": "assistant",
"content": "I have a Gigamuru that my uncle gave me as a gift. I love to play it at home."
},
{
    # Again user asks the question
"role": "user",
"content": "To 'screeg' something is to swing a sword at it. An example of a sentence that uses the word screeg is:"
}
]

# Now the model has learned tha pattern and now model has to respond to the new query

# Apply the chat template to the prompt that model expects
print(tokenizer.apply_chat_template(one_shot_prompt, tokenize=False))


# Generate the output
outputs = pipe(one_shot_prompt)
print(outputs[0]["generated_text"])

In [ ]:
#                      Chain Prompting: Breaking up the Problem
#              The model focuses on one task at a time, which often gives better results

In [ ]:
# Create name and slogan for a product

product_prompt = [
{"role": "user", "content": "Create a name and slogan for a chatbot that leverages LLMs."}
]

outputs = pipe(product_prompt)
product_description = outputs[0]["generated_text"]
print(product_description)

In [ ]:
# Based on a name and slogan for a product, generating a sales pitch

sales_prompt = [
{"role": "user",
 "content": f"Generate a very short sales pitch for the following product: '{product_description}'"}
]

outputs = pipe(sales_prompt)
sales_pitch = outputs[0]["generated_text"]

print(sales_pitch)

In [ ]:
"""    Reasoning with Generative Models
Chain of thought = Thinking step by step before giving an answer.
                   Solves the problem step by step 
                   """

In [ ]:
# Few shot chain of thought
# We are providing the example that shows the reasoning style to assistant
# so that it can answer the next query as it learned pattern
cot_prompt = [
    {
        "role": "user",
        "content": (
            "Roger has 5 tennis balls. He buys 2 more cans of tennis balls. "
            "Each can has 3 tennis balls. How many tennis balls does he have now?"
        ),
    },
    {
        "role": "assistant",
        "content": (
            "Roger started with 5 balls. "
            "2 cans of 3 tennis balls each is 6 tennis balls. "
            "5 + 6 = 11. The answer is 11."
        ),
    },
    {
        "role": "user",
        "content": (
            "The cafeteria had 23 apples. "
            "If they used 20 to make lunch and bought 6 more, "
            "how many apples do they have?"
        ),
    },
]

# Generate the output
outputs = pipe(cot_prompt)

# Print the generated text
print(outputs[0]["generated_text"])

In [ ]:
# Zero-shot chain-of-thought
# In zeroshot we doesnt have examples So use " Think step by step "

zeroshot_cot_prompt = [
{"role": "user", "content": "The cafeteria had 23 apples. If they used 20 to make lunch and bought 6 more, how many apples do they have? Let's think step-by-step."}
]

# Generate the output
outputs = pipe(zeroshot_cot_prompt)
print(outputs[0]["generated_text"])

In [ ]:
# TREE OF THOUGHT
# Instead of following only one reasoning path,
# the model explores multiple possible reasoning paths and
# chooses the best one before continuing.

In [ ]:
# Zero-shot Tree-of-Thought prompt
zeroshot_tot_prompt = [
    {
        "role": "user",
        "content": (
            "Imagine three different experts are answering this question. "
            "All experts will write down 1 step of their thinking, then share it "
            "with the group. Then all experts will go on to the next step, etc. "
            "If any expert realizes they're wrong at any point then they leave. "
            "The question is: 'The cafeteria had 23 apples. If they used 20 to "
            "make lunch and bought 6 more, how many apples do they have?' "
            "Make sure to discuss the results."
        ),
    }
]

# Generate the output
outputs = pipe(zeroshot_tot_prompt)

# Print the generated text
print(outputs[0]["generated_text"])

In [ ]:
           #  OUTPUT VERIFICATION

In [ ]:
# Checcking t=wether the LLM answer is in correct format or not
# It is a result checking step

In [ ]:
# Zero-shot learning: Providing no examples

# Here we wan the output in json format
zeroshot_prompt = [
{"role": "user", "content": "Create a character profile for an RPG game in JSON format."}
]
# Generate the output
outputs = pipe(zeroshot_prompt)
print(outputs[0]["generated_text"])

In [ ]:
# One-shot learning: Providing an example of the output structure

# Model gives the response as same as the instruction provided
one_shot_template = """Create a short character profile for an
RPG game. Make sure to only use this format:
{
"description": "A SHORT DESCRIPTION",
"name": "THE CHARACTER'S NAME",
"armor": "ONE PIECE OF ARMOR",
"weapon": "ONE OR MORE WEAPONS"
}
"""
one_shot_prompt = [
{"role": "user", "content": one_shot_template}
]
# Generate the output
outputs = pipe(one_shot_prompt)
print(outputs[0]["generated_text"])

In [ ]:
                    # Grammar: Constrained Sampling

# It forces the model to follow our instructions for sure


In [ ]:
# Import Llama class from llama.cpp.llama library which helps to load and run the GGUF models
from llama_cpp.llama import Llama
# Load Phi-3
llm = Llama.from_pretrained(
    # it downloads the GGUF version of Phi-3 Mini 4K Instruct.
repo_id="microsoft/Phi-3-mini-4k-instruct-gguf",

    # load the gguf file endig with fp16.gguf
filename="*fp16.gguf",

    # Decides how many model layers should run on the GPU.
n_gpu_layers=-1,

    # context window size
n_ctx=2048,

    # Hide extra information.
verbose=False
)

In [ ]:
# Generate output

# llm.create_chat_completion = calls the chat completion function from llm
# It tells the llm to generate response for messages
# The input format is different

output = llm.create_chat_completion(
messages=[
{"role": "user", "content": "Create a warrior for an RPG in JSON format."},
],
      # The output should be in json format
response_format={"type": "json_object"},
    temperature=0,    # Generate the response with as little randomness

)['choices'][0]['message']["content"]

# The response has a key called choices
# choices contains a list of generated choices.
# [0] = Select the first item in the choices list.
# ['choices'][0]['message'] = this access message key
# ['choices'][0]['message']["content"] =  the messages contains ai response in content

In [ ]:
import json
# python built in modelue work for json data
# Format as json
# json.loads(output) = takes the JSON text stored in output and
# converts it into a Python object, usually a dictionary.

# dumps() does the opposite of loads().
# converts the python objects inot json format
               # when we dump if the output is not json it will raise an error (CHECKING PURPOSE)
json_output = json.dumps(json.loads(output), indent=4)
print(json_output)